# Fundamentos de LCEL (LangChain Expression Language)

## ¿Qué es LCEL?

**LCEL (LangChain Expression Language)** es la forma moderna y recomendada de construir aplicaciones con LangChain. Sustituye a las "Chains" antiguas con una sintaxis más limpia, composable y basada en operadores.

### Ventajas de LCEL sobre las Chains antiguas:

1. **Sintaxis más limpia**: Usa el operador pipe (`|`) en lugar de métodos encadenados
2. **Composición fácil**: Puedes combinar componentes de forma intuitiva
3. **Mejor tipado**: Soporte completo de type hints
4. **Lazy evaluation**: Los componentes se ejecutan solo cuando es necesario
5. **Streaming nativo**: Soporte integrado para respuestas en tiempo real
6. **Debugging mejorado**: Mejor trazabilidad y manejo de errores

### Conceptos clave:

- **Runnable**: Cualquier componente que puede ser "ejecutado" (modelos, prompts, parsers, etc.)
- **Pipe (`|`)**: Operador que conecta Runnables en secuencia
- **Invoke**: Método para ejecutar un Runnable de forma síncrona
- **Stream**: Método para obtener respuestas en tiempo real

En este notebook aprenderás los fundamentos paso a paso.


In [1]:
# Configuración inicial
import sys
import os

# Hack para importar desde src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import get_local_llm

print("✓ Configuración completada")


✓ Configuración completada


## Lección 1: Modelos (The Runnable)

El modelo LLM es el componente más básico. En LCEL, los modelos son **Runnables**, lo que significa que pueden ser ejecutados directamente o conectados con otros componentes usando el operador pipe (`|`).

### Instanciar el modelo

Usamos nuestra función `get_local_llm()` que devuelve un `ChatOllama` configurado.


In [2]:
# Instanciar el modelo local
llm = get_local_llm()

print(f"Tipo del modelo: {type(llm)}")
print(f"Modelo configurado: {llm.model}")


Tipo del modelo: <class 'langchain_community.chat_models.ollama.ChatOllama'>
Modelo configurado: llama3


h:\Mi unidad\repositories_github\modern-local-langchain\src\models.py:19: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


### Invocar el modelo directamente

El método `.invoke()` ejecuta el modelo de forma síncrona. **Importante**: Para modelos de chat, necesitamos pasar una **lista de mensajes**, no un solo mensaje. Esto permite mantener conversaciones con múltiples mensajes.


In [5]:
# Invocar el modelo con un mensaje simple
from langchain_core.messages import HumanMessage

# IMPORTANTE: invoke() necesita una LISTA de mensajes, no un solo mensaje
mensaje = HumanMessage(content="Hola, ¿cómo estás?")
respuesta = llm.invoke([mensaje])  # Pasar como lista

print("Respuesta completa:")
print(respuesta)
print("\n" + "="*60)
print(f"Tipo de respuesta: {type(respuesta)}")
print(f"Contenido: {respuesta.content}")


Respuesta completa:
content='Hola! Como soy una inteligencia artificial, no tengo sentimientos ni emociones como los seres humanos, así que no estoy realmente "bien" o "mal". Estoy simplemente aquí para ayudarte y responder a tus preguntas al mejor de mis alcances. ¿En qué puedo ayudarte hoy?' additional_kwargs={} response_metadata={'model': 'llama3', 'created_at': '2025-11-21T22:32:00.2128367Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 27997440700, 'load_duration': 14583891900, 'prompt_eval_count': 18, 'prompt_eval_duration': 1635229100, 'eval_count': 66, 'eval_duration': 10787420100} id='lc_run--860b8649-2c43-4378-baef-120292bf2396-0'

Tipo de respuesta: <class 'langchain_core.messages.ai.AIMessage'>
Contenido: Hola! Como soy una inteligencia artificial, no tengo sentimientos ni emociones como los seres humanos, así que no estoy realmente "bien" o "mal". Estoy simplemente aquí para ayudarte y responder a tus preguntas al 

### Estructura del AIMessage

La respuesta es un objeto `AIMessage` que contiene:

- **`content`**: El texto generado por el modelo
- **`response_metadata`**: Metadatos adicionales (tokens usados, modelo, etc.)
- **`id`**: Identificador único del mensaje
- **`usage_metadata`**: Información sobre el uso de tokens (si está disponible)

Exploremos estos atributos:


In [6]:
# Explorar la estructura del AIMessage
print("Atributos del AIMessage:")
print(f"  - content: {respuesta.content}")
print(f"  - id: {respuesta.id}")
print(f"\n  - response_metadata:")
for key, value in respuesta.response_metadata.items():
    print(f"      {key}: {value}")
    
if hasattr(respuesta, 'usage_metadata') and respuesta.usage_metadata:
    print(f"\n  - usage_metadata:")
    for key, value in respuesta.usage_metadata.items():
        print(f"      {key}: {value}")


Atributos del AIMessage:
  - content: Hola! Como soy una inteligencia artificial, no tengo sentimientos ni emociones como los seres humanos, así que no estoy realmente "bien" o "mal". Estoy simplemente aquí para ayudarte y responder a tus preguntas al mejor de mis alcances. ¿En qué puedo ayudarte hoy?
  - id: lc_run--860b8649-2c43-4378-baef-120292bf2396-0

  - response_metadata:
      model: llama3
      created_at: 2025-11-21T22:32:00.2128367Z
      message: {'role': 'assistant', 'content': ''}
      done: True
      done_reason: stop
      total_duration: 27997440700
      load_duration: 14583891900
      prompt_eval_count: 18
      prompt_eval_duration: 1635229100
      eval_count: 66
      eval_duration: 10787420100


## Lección 2: Prompts

Los **Prompts** son plantillas que formatean el texto antes de enviarlo al modelo. En LCEL, usamos `ChatPromptTemplate` para crear prompts estructurados.

### Crear un Prompt Template

Un template permite definir variables que se rellenan dinámicamente.


In [7]:
from langchain_core.prompts import ChatPromptTemplate

# Crear un template de prompt
template = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor experto."),
    ("human", "Traduce esto al {idioma}: {texto}")
])

print("Template creado:")
print(template)


Template creado:
input_variables=['idioma', 'texto'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Eres un traductor experto.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['idioma', 'texto'], input_types={}, partial_variables={}, template='Traduce esto al {idioma}: {texto}'), additional_kwargs={})]


### Invocar el Prompt (sin modelo)

**Importante**: Cuando invocas solo el prompt con `.invoke()`, solo se formatea el texto. **NO se llama al modelo**. Esto es útil para ver cómo se verá el mensaje antes de enviarlo.


In [ ]:
# Invocar solo el prompt (sin modelo)
mensaje_formateado = template.invoke({
    "idioma": "inglés",
    "texto": "Hola, ¿cómo estás?"
})

print("Mensaje formateado (sin llamar al modelo):")
print(mensaje_formateado)
print(f"\nTipo: {type(mensaje_formateado)}")
print(f"\nMensajes:")
for msg in mensaje_formateado.messages:
    print(f"  - {type(msg).__name__}: {msg.content}")


## Lección 3: Output Parsers

Los **Output Parsers** extraen y transforman la salida del modelo. El más común es `StrOutputParser`, que simplemente extrae el texto del `AIMessage`.

### ¿Por qué necesitamos parsers?

Cuando el modelo devuelve un `AIMessage`, a menudo solo necesitamos el texto. Los parsers nos permiten extraer solo lo que necesitamos de forma limpia.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Crear el parser
output_parser = StrOutputParser()

print(f"Tipo del parser: {type(output_parser)}")

# Ejemplo: extraer texto de un AIMessage
# (usando la respuesta anterior)
texto_limpio = output_parser.invoke(respuesta)

print(f"\nTexto extraído: {texto_limpio}")
print(f"Tipo: {type(texto_limpio)}")


## Lección 4: The Pipe (`|`) - La Magia de LCEL

El operador pipe (`|`) es el corazón de LCEL. Permite conectar Runnables en secuencia de forma elegante y legible.

### Sintaxis

```python
chain = prompt | model | output_parser
```

Esto significa:
1. El **prompt** formatea el input
2. El **modelo** genera la respuesta
3. El **parser** extrae el texto limpio

### Crear una cadena completa


In [ ]:
# Crear la cadena completa usando el pipe
chain = template | llm | output_parser

print("Cadena creada:")
print(f"  Tipo: {type(chain)}")
print(f"  Componentes: prompt -> model -> parser")


### Ejecutar la cadena

Ahora podemos invocar la cadena completa pasando solo el diccionario con las variables del prompt. La cadena se encarga de:
1. Formatear el prompt
2. Llamar al modelo
3. Extraer el texto


In [ ]:
# Ejecutar la cadena completa
resultado = chain.invoke({
    "idioma": "francés",
    "texto": "Buenos días, ¿cómo puedo ayudarte?"
})

print("Resultado de la traducción:")
print(resultado)
print(f"\nTipo del resultado: {type(resultado)}")


### Ventajas del Pipe

1. **Legibilidad**: El código es más fácil de leer y entender
2. **Composición**: Puedes agregar o quitar componentes fácilmente
3. **Reutilización**: Cada componente puede usarse en múltiples cadenas
4. **Debugging**: Puedes invocar cada componente individualmente para debuggear

### Ejemplo: Agregar más componentes

Puedes encadenar tantos componentes como necesites:


In [ ]:
# Ejemplo: Cadena más compleja
# Podríamos agregar más pasos, como un formateador adicional

# Cadena simple: prompt -> model -> parser
chain_simple = template | llm | output_parser

# Ejecutar con otro ejemplo
resultado2 = chain_simple.invoke({
    "idioma": "italiano",
    "texto": "Me encanta programar con Python"
})

print("Traducción al italiano:")
print(resultado2)


## Resumen de Conceptos Clave

1. **Runnable**: Cualquier componente ejecutable (modelos, prompts, parsers)
2. **`.invoke()`**: Ejecuta un Runnable de forma síncrona
3. **Pipe (`|`)**: Conecta Runnables en secuencia
4. **ChatPromptTemplate**: Crea prompts con variables
5. **StrOutputParser**: Extrae texto limpio de AIMessage

### Flujo típico de LCEL:

```
Input (dict) 
  → Prompt (formatea) 
  → Model (genera AIMessage) 
  → Parser (extrae texto) 
  → Output (str)
```


## 🎯 Ejercicio Práctico

### Tu Turno

Crea una cadena LCEL que:

1. Tome un **tema** como input
2. Genere un **tweet corto y divertido** sobre ese tema usando Llama 3
3. Devuelva solo el texto del tweet

**Requisitos:**
- Usa `ChatPromptTemplate` para crear el prompt
- El prompt debe pedirle al modelo que genere un tweet divertido sobre el tema
- Usa el operador pipe (`|`) para conectar: prompt → modelo → parser
- Ejecuta la cadena con al menos 2 temas diferentes

**Pistas:**
- El template del prompt podría ser algo como: "Escribe un tweet corto y divertido sobre: {tema}"
- Recuerda usar `StrOutputParser` para obtener solo el texto
- Usa `.invoke()` con un diccionario que contenga el tema

¡Buena suerte! 🚀


In [ ]:
# Tu código aquí
# Crea tu cadena LCEL para generar tweets divertidos

